In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, date, timedelta

# =========================
# 1) DB 설정 (사양 그대로)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

SCHEMA = "e4_predictive_maintenance"
TABLE_PD = "pd_worst"
TABLE_NONPD = "non_pd_worst"

def get_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        future=True,
    )

engine = get_engine()

# =========================
# 2) 입력
# =========================
PROD_DAY = "2026-01-08"   # "YYYY-MM-DD"
SHIFT_TYPE = "both"       # "day" | "night" | "both"


def _parse_prod_day(prod_day: str) -> date:
    return datetime.strptime(prod_day, "%Y-%m-%d").date()


def _build_shift_where(prod_day: str, shift_type: str):
    """
    end_day(date) + end_time(text)로 주/야간 시간창 필터.
    end_time(text)는 "HH:MI:SS"로 정규화(소수점 이하 제거) 후 time 캐스팅.
    """
    d0 = _parse_prod_day(prod_day)
    d1 = d0 + timedelta(days=1)

    # end_time 정규화: trim + 소수점 이하 제거
    norm_end_time_sql = r"regexp_replace(btrim(end_time), '\..*$', '')"

    if shift_type == "day":
        where_sql = f"""
        (end_day = %(d0)s
         AND ({norm_end_time_sql})::time BETWEEN '08:30:00'::time AND '20:29:59'::time)
        """
        params = {"d0": d0}
        return where_sql, params, norm_end_time_sql

    if shift_type == "night":
        where_sql = f"""
        (
          (end_day = %(d0)s AND ({norm_end_time_sql})::time BETWEEN '20:30:00'::time AND '23:59:59'::time)
          OR
          (end_day = %(d1)s AND ({norm_end_time_sql})::time BETWEEN '00:00:00'::time AND '08:29:59'::time)
        )
        """
        params = {"d0": d0, "d1": d1}
        return where_sql, params, norm_end_time_sql

    raise ValueError("shift_type must be 'day' or 'night'")


def fetch_worst_list(prod_day: str, shift_type: str) -> pd.DataFrame:
    """
    pd_worst / non_pd_worst를 합쳐서 리턴.
    remark 라벨:
      - pd_worst -> 'PD'
      - non_pd_worst -> 'Non-PD'
    prod_day/shift_type 라벨은 파이썬에서 컬럼으로 추가.
    """
    assert shift_type in ("day", "night"), "shift_type must be 'day' or 'night'"

    where_sql, params, norm_end_time_sql = _build_shift_where(prod_day, shift_type)

    sql = f"""
    WITH base AS (
        SELECT
            barcode_information,
            'PD'::text    AS remark,
            station,
            end_day,
            ({norm_end_time_sql}) AS end_time_norm,
            run_time,
            test_contents,
            file_path
        FROM {SCHEMA}.{TABLE_PD}
        WHERE {where_sql}

        UNION ALL

        SELECT
            barcode_information,
            'Non-PD'::text AS remark,
            station,
            end_day,
            ({norm_end_time_sql}) AS end_time_norm,
            run_time,
            test_contents,
            file_path
        FROM {SCHEMA}.{TABLE_NONPD}
        WHERE {where_sql}
    )
    SELECT
        barcode_information,
        remark,
        station,
        run_time,
        test_contents,
        end_day,
        end_time_norm AS end_time,
        file_path
    FROM base
    ORDER BY end_day DESC, end_time_norm DESC, station ASC;
    """

    with engine.connect() as conn:
        df = pd.read_sql_query(sql, conn, params=params)

    # 라벨 컬럼은 파이썬에서 추가 (DB 파라미터 캐스팅 불필요)
    df.insert(0, "shift_type", shift_type)
    df.insert(0, "prod_day", _parse_prod_day(prod_day))

    return df


# =========================
# 3) 실행
# =========================
if SHIFT_TYPE == "day":
    df_day = fetch_worst_list(PROD_DAY, "day")
    display(df_day[["prod_day","shift_type","barcode_information","remark","station","run_time","test_contents"]])

elif SHIFT_TYPE == "night":
    df_night = fetch_worst_list(PROD_DAY, "night")
    display(df_night[["prod_day","shift_type","barcode_information","remark","station","run_time","test_contents"]])

elif SHIFT_TYPE == "both":
    df_day = fetch_worst_list(PROD_DAY, "day")
    df_night = fetch_worst_list(PROD_DAY, "night")

    print("=== DAY ===")
    display(df_day[["prod_day","shift_type","barcode_information","remark","station","run_time","test_contents","file_path"]])

    print("=== NIGHT ===")
    display(df_night[["prod_day","shift_type","barcode_information","remark","station","run_time","test_contents","file_path"]])


else:
    raise ValueError("SHIFT_TYPE must be 'day', 'night', or 'both'")

=== DAY ===


,prod_day,shift_type,barcode_information,remark,station,run_time,test_contents,file_path
0,2026-01-08,day,BA1WJ26008500071USJ8T-14F014-AF,PD,FCT1,50.1,1.23_overcurr_usb_a_v,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
1,2026-01-08,day,BA1WJ26008503370USJ8T-14F014-AF,PD,FCT1,55.8,1.28_usb_c_bside_3c_check,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
2,2026-01-08,day,BA1WJ26008503072USJ8T-14F014-AF,PD,FCT1,51.3,1.28_load_c_3c_set,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
3,2026-01-08,day,BA1WJ26008502096USJ8T-14F014-AF,PD,FCT1,47.0,1.12_usb_c_carplay,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...


=== NIGHT ===


,prod_day,shift_type,barcode_information,remark,station,run_time,test_contents,file_path
0,2026-01-08,night,BA1WJ26009500838USJ8T-14F014-AF,PD,FCT1,45.4,1.01_ps_14.7_on,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
1,2026-01-08,night,BA1WJ26009500838USJ8T-14F014-AF,PD,FCT1,45.4,1.00_load_c_cc_rng_set,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
